# New SROIE C2 validation: 350 documents
Upload the original test ZIP, newly rendered ZIP and full metadata ZIP to MyDrive/SROIE_Current_C2. Use a GPU runtime. This evaluates a NEW zero-margin policy, not historical Table 6. Strong and RA use identical images, so one shared masked inference is sufficient. Raw outputs and per-field exact matches are saved. The model revision is recorded, but historical revision equivalence must be checked before claiming exact historical reproducibility.


In [ ]:
!pip -q install "transformers>=4.46,<5" sentencepiece accelerate pandas scipy
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import io,json,zipfile,re,hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import DonutProcessor,VisionEncoderDecoderModel
BASE=Path('/content/drive/MyDrive/SROIE_Current_C2')
ORIGINAL=BASE/'SROIE-PII-full971-images-test.zip'
MASKED=BASE/'SROIE_Current_C2_Rendered350.zip'
METADATA=BASE/'SROIE-PII-full971-metadata.zip'
OUT=BASE/'Results';OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available(),'Select GPU runtime'
with zipfile.ZipFile(METADATA) as z:docs=[d for d in json.loads(z.read('sroie_pii_full971.json'))['documents'] if d['split']=='test']
assert len(docs)==350
MODEL='philschmid/donut-base-sroie'
processor=DonutProcessor.from_pretrained(MODEL)
model=VisionEncoderDecoderModel.from_pretrained(MODEL).to('cuda').eval()
revision=getattr(model.config,'_commit_hash',None)
prompt=processor.tokenizer('<s>',add_special_tokens=False,return_tensors='pt').input_ids.to('cuda')
fields=['company','date','address','total']
def norm(s):return re.sub('[^a-z0-9]','',str(s or '').casefold())
records=[]
with zipfile.ZipFile(ORIGINAL) as zo,zipfile.ZipFile(MASKED) as zm:
 for condition,z in [('ORIGINAL',zo),('MASKED_SHARED',zm)]:
  for d in docs:
   name=d['image_file'] if condition=='ORIGINAL' else 'shared_Strong_RA_C2/'+d['document_id']+'.png'
   im=Image.open(io.BytesIO(z.read(name))).convert('RGB')
   pixels=processor(im,return_tensors='pt').pixel_values.to('cuda')
   with torch.inference_mode():
    seq=model.generate(pixels,decoder_input_ids=prompt,max_length=model.decoder.config.max_position_embeddings,early_stopping=True,pad_token_id=processor.tokenizer.pad_token_id,eos_token_id=processor.tokenizer.eos_token_id,use_cache=True,num_beams=1,bad_words_ids=[[processor.tokenizer.unk_token_id]])
   raw=processor.batch_decode(seq)[0]
   cleaned=raw.replace(processor.tokenizer.eos_token or '', '').replace(processor.tokenizer.pad_token or '', '').strip()
   try:pred=processor.token2json(cleaned)
   except Exception:pred={}
   if not isinstance(pred,dict):pred={}
   r={'document_id':d['document_id'],'condition':condition,'raw_prediction':raw}
   exact=[]
   for f in fields:
    gt=d['source_entities'][f];pv=pred.get(f,'')
    r['gt_'+f]=gt;r['pred_'+f]=str(pv);r['exact_'+f]=int(norm(gt)==norm(pv));exact.append(r['exact_'+f])
   r['field_exact']=float(np.mean(exact));records.append(r)
   pd.DataFrame(records).to_csv(OUT/'checkpoint_predictions.csv',index=False)
   if len(records)%25==0:print(len(records),'/700')
df=pd.DataFrame(records);assert len(df)==700
df.to_csv(OUT/'predictions.csv',index=False)
summary=df.groupby('condition')['field_exact'].agg(['mean','count']);summary.to_csv(OUT/'summary.csv')
piv=df.pivot(index='document_id',columns='condition',values='field_exact')
delta=(piv.MASKED_SHARED-piv.ORIGINAL).to_numpy()
rng=np.random.default_rng(20261003)
boot=np.array([rng.choice(delta,len(delta),replace=True).mean() for _ in range(10000)])
audit={'model':MODEL,'resolved_revision':revision,'documents':350,'actual_inferences':700,'Strong_RA_outputs':'identical images; shared inference','field_metric':'mean normalized field exact match; not standard token F1','masked_minus_original':float(delta.mean()),'ci95':np.quantile(boot,[.025,.975]).tolist(),'Strong_minus_RA':0.0,'note':'No training or tuning. G2/G3 contextual benefits are not evaluated.'}
(OUT/'audit.json').write_text(json.dumps(audit,indent=2))
print(summary);print(audit)
